# Q-Learning Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/adversarial2dEnvAI/blob/master/notebooks/Q_Learning.ipynb)

In diesem Notebook implementieren wir **Q-Learning**, einen modell-freien Reinforcement Learning Algorithmus. Im Gegensatz zu Value Iteration muss der Agent hier nicht wissen, wie die Umgebung funktioniert (Transitionen/Rewards), sondern lernt durch Erfahrung.

## 1. Setup

In [ ]:
!pip install git+https://github.com/dgaida/adversarial2dEnvAI.git
import os
import numpy as np
import matplotlib.pyplot as plt
from custom_grid_env.interface import AgentInterface
from custom_grid_env.agents.q_learning_agent import QLearningAgent

# Set dummy video driver for headless environment
os.environ["SDL_VIDEODRIVER"] = "dummy"

## 2. Der Algorithmus: Q-Learning

Die Update-Regel lautet:
$$Q(s, a) \\leftarrow Q(s, a) + \\alpha [r + \\gamma \\max_{a'} Q(s', a') - Q(s, a)]$$

Wichtige Hyperparameter:
- $\\alpha$ (Alpha): Learning Rate
- $\\gamma$ (Gamma): Discount Factor
- $\\epsilon$ (Epsilon): Exploration Rate (Epsilon-Greedy Policy)

In [ ]:
interface = AgentInterface(render=True, render_mode="rgb_array", slip_probability=0.1)
agent = QLearningAgent(interface.get_action_space(), alpha=0.1, gamma=0.9, epsilon=0.3)

episodes = 500
rewards_history = []

for ep in range(episodes):
    obs = interface.reset()
    state = tuple(interface.env.agent_pos)
    total_reward = 0
    done = False
    
    while not done:
        action = agent.get_action(obs)
        obs, reward, done, info = interface.step(action)
        next_state = tuple(interface.env.agent_pos)
        
        agent.update(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
    
    rewards_history.append(total_reward)
    if (ep + 1) % 50 == 0:
        # Epsilon Decay
        agent.epsilon *= 0.95
        print(f"Episode {ep+1}/{episodes} - Avg Reward: {np.mean(rewards_history[-50:]):.1f}")

## 3. Analyse des Lernfortschritts

In [ ]:
plt.plot(rewards_history)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Training Progress")
plt.show()

## 4. Visualisierung der Q-Table

Wir zeigen den maximalen Q-Wert pro Zelle.

In [ ]:
q_img = np.zeros((interface.env.rows, interface.env.cols))
for r in range(interface.env.rows):
    for c in range(interface.env.cols):
        q_img[r, c] = agent.get_value((r, c))

plt.imshow(q_img, cmap='magma')
plt.colorbar(label='Max Q-Value')
plt.title("Learned Q-Function")
plt.show()